# <font color='blue'> Lab-05 (Solution) <font>

### <font color='blue'> Problem 1: Data Preparation <font>
    
(1.0) Download any customer churn dataset of your choice.
    
(1.1) Use Pandas to load the dataset and display the first few rows of the dataset.
    
(1.2) Calculate the frequency of churn and no churn labels in the dataset to assess the data imbalance.
    
<font color='brown'> Hint: You can download the data from here: https://huggingface.co/datasets/scikit-learn/churn-prediction/tree/main <font>

In [174]:
# (1.1)
import pandas as pd
data = pd.read_csv('https://huggingface.co/datasets/scikit-learn/churn-prediction/raw/main/dataset.csv')
print(data.head(2))

#(1.2)
check_balance = data['Churn'].value_counts()
print("\n Checking for data imbalance:\n",check_balance,"\n")

   customerID  gender  SeniorCitizen Partner Dependents  tenure PhoneService  \
0  7590-VHVEG  Female              0     Yes         No       1           No   
1  5575-GNVDE    Male              0      No         No      34          Yes   

      MultipleLines InternetService OnlineSecurity  ... DeviceProtection  \
0  No phone service             DSL             No  ...               No   
1                No             DSL            Yes  ...              Yes   

  TechSupport StreamingTV StreamingMovies        Contract PaperlessBilling  \
0          No          No              No  Month-to-month              Yes   
1          No          No              No        One year               No   

      PaymentMethod MonthlyCharges  TotalCharges Churn  
0  Electronic check          29.85         29.85    No  
1      Mailed check          56.95        1889.5    No  

[2 rows x 21 columns]

 Checking for data imbalance:
 No     5174
Yes    1869
Name: Churn, dtype: int64 



### <font color='blue'> Problem 2: Data Pre-processing <font>
    
   
(2.1) Check for missing values in the dataset and handle them if any.
    
(2.2) Convert the Churn column to binary (0 for 'No', 1 for 'Yes') using label encoding.
    
Preview the data to ensure pre-processing is as expected.
    

In [139]:
# (2.1)
from sklearn.impute import SimpleImputer


missed_values = data.isnull().sum()
if missed_values.sum() > 0:
    print("There are missing values in the data.\n")
    method = SimpleImputer(strategy='most_frequent')  # Or 'median', 'most_frequent'
    data = method.fit_transform(data)
    print("Missing Values handled!\n")
else:
    print("There are no missing values in the data.\n")


# (2.2)
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
data['Churn'] = le.fit_transform(data['Churn'])

# Preview the data to ensure pre-processing is as expected
data.head()

There are no missing values in the data.



,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,0
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,0
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,1
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,0
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,1


### <font color='blue'> Problem 3: Feature Engineering and Data split <font>
    
(3.1) Identify and list the categorical (text or mixed types of data) and numerical features in the dataset.

(3.2) Convert categorical variables into dummy/indicator variables.

(3.3) Standardize the numerical features to have a mean of 0 and a standard deviation of 1.
    
(3.4) Split the dataset into training (80%) and testing (20%) sets.
    
 (3.4*) (Optional) Check the shape of the train and test sets



In [176]:
#(3.1)

# For categorical variables select columns with the data type 'object' (containing text or mixed types of data)
category = data.select_dtypes(include=['object'])

# Get the column names as a list
f_cat = category.columns.tolist()
print(f"List of {len(f_cat)} Categorical Features:\n {f_cat}")


####### TotalCharges #######
"""
This feature was problematic during the lab session as its values are stored as objects (strings), though they should be stored as numericals
In addition to this, the feature has several empty strings (' ') making it difficult to detect them as missing values (this is also the reason the cell above did not display any missing values)
Thus, the code below zero imputes for empty strings and transforms the strings into floats otherwise
"""
data['TotalCharges'] = [0 if value == ' ' else float(value) for value in data['TotalCharges']]
##############################

f_num = data.select_dtypes(exclude=['object']).columns.tolist()
print(f"\n List of {len(f_num)} Numerical Features:\n {f_num}")

#(3.2)
from sklearn.preprocessing import LabelEncoder, StandardScaler

# Selecting relevant features and output variable
X = data.drop(['customerID', 'Churn'], axis=1) # Dropping 'customerID' as it's not relevant for prediction
y = data['Churn']


#(3.3)
# Converting categorical variables into dummy/indicator variables
X = pd.get_dummies(X) # This method does not transform non-numerical values

# Standardizing the features
scl = StandardScaler()
X_scaled = scl.fit_transform(X)

#(3.4)
from sklearn.model_selection import train_test_split

# Splitting the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

#  (3.4*) (Optional) Checking the shape of the training and testing sets
#shape- return (number of rows (samples) , the number of columns (features))
X_train.shape, X_test.shape, y_train.shape, y_test.shape

List of 18 Categorical Features:
 ['customerID', 'gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'TotalCharges', 'Churn']

 List of 4 Numerical Features:
 ['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']


((5634, 45), (1409, 45), (5634,), (1409,))

###  <font color='blue'> Problem 4: Logistic Regression Model Training <font>
    
(4.1) Train a logistic regression model using the training set in scikit-learn.
     
(4.2) Predict the churn outcome on the testing set.

In [172]:
from sklearn.linear_model import LogisticRegression

# (4.1)
lrm = LogisticRegression(max_iter=1000)
lrm.fit(X_train, y_train)

# (4.2) System Predicted outcome
y_hat = lrm.predict(X_test)

### <font color='blue'> Problem 5: Model Evaluation <font>
    

         
(5.1) Calculate the accuracy of the model.
     
(5.2) Calculate precision, recall, F1-score for both classes (1 and 0) and print them in table format.
     
(5.3) Generate and interpret a confusion matrix.

In [173]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import pandas as pd

# (5.1) Accuracy
accuracy = accuracy_score(y_test, y_hat)
print(accuracy)

# (5.2) Precision, Recall, and F1-score for both classes (1 and 0)
pre_0 = precision_score(y_test, y_hat, pos_label=0)
pre_1 = precision_score(y_test, y_hat, pos_label=1)
r_0 = recall_score(y_test, y_hat, pos_label=0)
r_1 = recall_score(y_test, y_hat, pos_label=1)
f_0 = f1_score(y_test, y_hat, pos_label=0)
f_1 = f1_score(y_test, y_hat, pos_label=1)

# Store precision, recall, and F1-score in a DataFrame
Evaluation = pd.DataFrame({
    ' ': ['Precision', 'Recall', 'F1-Score'],
    'No Churn(0)': [round(pre_0,2), round(r_0,2), round(f_0,2)],
    'Churn(1)': [round(pre_1,2), round(r_1,2), round(f_1,2)]
})
print("\n Precision, Recall and F1-score for both classes\n")
print(Evaluation,"\n")

# (Step )5.3) Confusion matrix
conf_matrix = confusion_matrix(y_hat, y_test)
print("Confusion Matrix \n", conf_matrix)


# Alternate way for (5.2)
from sklearn.metrics import classification_report, confusion_matrix
print("\n Generating complete evaluation report....\n")
report = classification_report(y_test, y_hat, target_names=['No Churn', 'Churn'])
print(report)

0.8204400283889283

 Precision, Recall and F1-score for both classes

              No Churn(0)  Churn(1)
0  Precision         0.86      0.68
1     Recall         0.90      0.60
2   F1-Score         0.88      0.64 

Confusion Matrix 
 [[933 150]
 [103 223]]

 Generating complete evaluation report....

              precision    recall  f1-score   support

    No Churn       0.86      0.90      0.88      1036
       Churn       0.68      0.60      0.64       373

    accuracy                           0.82      1409
   macro avg       0.77      0.75      0.76      1409
weighted avg       0.81      0.82      0.82      1409

